# Advanced Problems: Python Default Values — Beware!

This notebook contains advanced problems with solutions on Python default argument evaluation, mutable defaults, sentinel objects, closures, introspection, and API design best practices.

## Key Rule

Default argument expressions are evaluated exactly once: when the function is defined, not each time the function is called.

In [1]:
from datetime import datetime
from time import sleep
from copy import deepcopy
import inspect

## Problem 1 — Frozen Timestamp Bug

The function below is intended to log the current UTC time whenever it is called. Explain the bug and fix it.

In [2]:
def bad_log(message, *, dt=datetime.utcnow()):
    return f'{dt}: {message}'

print(bad_log('first'))
sleep(1)
print(bad_log('second'))

C:\Users\user1\AppData\Local\Temp\ipykernel_22120\1691157001.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  def bad_log(message, *, dt=datetime.utcnow()):


2026-05-10 12:24:55.851414: first
2026-05-10 12:24:55.851414: second


### Solution 1

`datetime.utcnow()` is evaluated when `bad_log` is defined. Therefore every call without an explicit `dt` reuses the same timestamp.

Best practice: use `None` as the default and compute the dynamic value inside the function.

In [3]:
def good_log(message, *, dt=None):
    if dt is None:
        dt = datetime.utcnow()
    return f'{dt}: {message}'

print(good_log('first'))
sleep(1)
print(good_log('second'))
print(good_log('manual', dt='2001-01-01 00:00:00'))

C:\Users\user1\AppData\Local\Temp\ipykernel_22120\1882015929.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  dt = datetime.utcnow()


2026-05-10 12:25:02.510853: first
2026-05-10 12:25:03.512137: second
2001-01-01 00:00:00: manual


## Problem 2 — Mutable Default Cache Leak

The following function is supposed to add one item to a fresh list each time unless a list is explicitly supplied. Find the bug and fix it.

In [4]:
def append_item_bad(item, items=[]):
    items.append(item)
    return items

print(append_item_bad('a'))
print(append_item_bad('b'))
print(append_item_bad('c'))

['a']
['a', 'b']
['a', 'b', 'c']


### Solution 2

The list `[]` is created once at function definition time and then reused. This creates hidden shared state.

Use `None` and create the list inside the function.

In [5]:
def append_item_good(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items

print(append_item_good('a'))
print(append_item_good('b'))

existing = ['x']
print(append_item_good('y', existing))
print(existing)

['a']
['b']
['x', 'y']
['x', 'y']


## Problem 3 — When `None` Is a Valid Argument

Sometimes `None` is a meaningful user-provided value. Write a function `configure_timeout(timeout=...)` where:

- omitted argument means use default timeout of `30`
- `None` means no timeout
- any positive number is used directly
- non-positive numbers raise `ValueError`

### Solution 3

When `None` is a valid input, do not use `None` as the sentinel for omission. Use a private sentinel object.

In [6]:
_MISSING = object()

def configure_timeout(timeout=_MISSING):
    if timeout is _MISSING:
        return 30
    if timeout is None:
        return None
    if timeout <= 0:
        raise ValueError('timeout must be positive, None, or omitted')
    return timeout

print(configure_timeout())       # omitted -> 30
print(configure_timeout(None))   # explicit None -> no timeout
print(configure_timeout(10))     # explicit value -> 10

try:
    configure_timeout(0)
except ValueError as e:
    print(type(e).__name__, e)

30
None
10
ValueError timeout must be positive, None, or omitted


## Problem 4 — `dt = dt or datetime.utcnow()` Pitfall

The following implementation looks compact, but it can be wrong:

```python
dt = dt or datetime.utcnow()
```

Explain why and write a safer version.

### Solution 4

`dt or datetime.utcnow()` replaces every falsy value, not only missing values. That can accidentally replace valid values such as `0`, `''`, `False`, or custom falsy objects.

Prefer `is None` when `None` means omitted.

In [7]:
def log_safer(message, *, dt=None):
    if dt is None:
        dt = datetime.utcnow()
    return f'{dt}: {message}'

print(log_safer('normal'))
print(log_safer('explicit empty string kept', dt=''))
print(log_safer('explicit zero kept', dt=0))

2026-05-10 12:25:09.254549: normal
: explicit empty string kept
0: explicit zero kept


C:\Users\user1\AppData\Local\Temp\ipykernel_22120\1627491970.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  dt = datetime.utcnow()


## Problem 5 — Intentional Shared State

Mutable defaults are usually a bug, but sometimes they are intentionally used for caching.

Write a Fibonacci function that intentionally uses a default dictionary as a cache. Then discuss why this pattern should be used carefully.

### Solution 5

This works because the dictionary persists across calls. However, this creates hidden function-level state. In production code, prefer `functools.lru_cache` unless you specifically need manual control.

In [8]:
def fib(n, _cache={0: 0, 1: 1}):
    if n < 0:
        raise ValueError('n must be non-negative')
    if n not in _cache:
        _cache[n] = fib(n - 1) + fib(n - 2)
    return _cache[n]

print(fib(10))
print(fib(30))
print(fib.__defaults__)

55
832040
({0: 0, 1: 1, 2: 1, 3: 2, 4: 3, 5: 5, 6: 8, 7: 13, 8: 21, 9: 34, 10: 55, 11: 89, 12: 144, 13: 233, 14: 377, 15: 610, 16: 987, 17: 1597, 18: 2584, 19: 4181, 20: 6765, 21: 10946, 22: 17711, 23: 28657, 24: 46368, 25: 75025, 26: 121393, 27: 196418, 28: 317811, 29: 514229, 30: 832040},)


## Problem 6 — Safer Cache with `functools.lru_cache`

Rewrite the previous Fibonacci example using a clearer, standard-library caching tool.

### Solution 6

In [9]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib_cached(n):
    if n < 0:
        raise ValueError('n must be non-negative')
    if n < 2:
        return n
    return fib_cached(n - 1) + fib_cached(n - 2)

print(fib_cached(30))
print(fib_cached.cache_info())

832040
CacheInfo(hits=28, misses=31, maxsize=None, currsize=31)


## Problem 7 — Default Values and Closures

The following code creates functions in a loop. Predict the output, then fix it.

```python
funcs = []
for i in range(3):
    funcs.append(lambda: i)

print([f() for f in funcs])
```

### Solution 7

Closures capture the variable `i`, not its value at each iteration. After the loop ends, `i == 2`, so every lambda returns `2`.

A common fix is to bind the current value using a default argument.

In [10]:
bad_funcs = []
for i in range(3):
    bad_funcs.append(lambda: i)

print([f() for f in bad_funcs])

good_funcs = []
for i in range(3):
    good_funcs.append(lambda i=i: i)

print([f() for f in good_funcs])

[2, 2, 2]
[0, 1, 2]


## Problem 8 — Inspecting Defaults

Use Python introspection to prove that default arguments are stored on the function object.

### Solution 8

In [11]:
def example(a, b=[], *, c=datetime.utcnow()):
    pass

print('positional defaults:', example.__defaults__)
print('keyword-only defaults:', example.__kwdefaults__)
print('signature:', inspect.signature(example))

example.__defaults__[0].append('mutated')
print('after mutation:', example.__defaults__)

positional defaults: ([],)
keyword-only defaults: {'c': datetime.datetime(2026, 5, 10, 12, 25, 17, 40817)}
signature: (a, b=[], *, c=datetime.datetime(2026, 5, 10, 12, 25, 17, 40817))
after mutation: (['mutated'],)


C:\Users\user1\AppData\Local\Temp\ipykernel_22120\4046020036.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  def example(a, b=[], *, c=datetime.utcnow()):


## Problem 9 — API Design with Copying

Write `register_user(name, tags=None)` so that:

- if `tags` is omitted, the user gets an empty independent tag list
- if `tags` is provided, the stored tags are protected from future external mutation
- the returned user is a dictionary

### Solution 9

Using `list(tags)` copies the provided iterable into a new list, preventing accidental aliasing with the caller's original object.

In [12]:
def register_user(name, tags=None):
    if tags is None:
        tags = []
    else:
        tags = list(tags)
    return {'name': name, 'tags': tags}

external_tags = ['admin']
user = register_user('Ada', external_tags)
external_tags.append('mutated-outside')

print(user)
print(external_tags)

{'name': 'Ada', 'tags': ['admin']}
['admin', 'mutated-outside']


## Problem 10 — Nested Mutable Data

The previous solution protects against mutation of the outer list. But what if the default data is nested?

Fix the function below.

In [13]:
def make_profile_bad(name, preferences={'theme': {'mode': 'light'}}):
    return {'name': name, 'preferences': preferences}

a = make_profile_bad('Ada')
b = make_profile_bad('Grace')

a['preferences']['theme']['mode'] = 'dark'

print(a)
print(b)

{'name': 'Ada', 'preferences': {'theme': {'mode': 'dark'}}}
{'name': 'Grace', 'preferences': {'theme': {'mode': 'dark'}}}


### Solution 10

Use `None` for the default and create fresh nested data inside the function. If user-provided nested data should be isolated, use `deepcopy`.

In [14]:
def make_profile_good(name, preferences=None):
    if preferences is None:
        preferences = {'theme': {'mode': 'light'}}
    else:
        preferences = deepcopy(preferences)
    return {'name': name, 'preferences': preferences}

a = make_profile_good('Ada')
b = make_profile_good('Grace')

a['preferences']['theme']['mode'] = 'dark'

print(a)
print(b)

external = {'theme': {'mode': 'solarized'}}
c = make_profile_good('Linus', external)
external['theme']['mode'] = 'mutated outside'
print(c)
print(external)

{'name': 'Ada', 'preferences': {'theme': {'mode': 'dark'}}}
{'name': 'Grace', 'preferences': {'theme': {'mode': 'light'}}}
{'name': 'Linus', 'preferences': {'theme': {'mode': 'solarized'}}}
{'theme': {'mode': 'mutated outside'}}


## Problem 11 — Advanced Debugging Challenge

The following function seems to behave randomly across calls. Diagnose the exact reason.

In [15]:
def collect_events_bad(event, history=[]):
    history.append({'time': datetime.utcnow(), 'event': event})
    return history

print(collect_events_bad('start'))
print(collect_events_bad('stop'))
print(collect_events_bad('restart'))
print(collect_events_bad.__defaults__)

[{'time': datetime.datetime(2026, 5, 10, 12, 25, 23, 370115), 'event': 'start'}]
[{'time': datetime.datetime(2026, 5, 10, 12, 25, 23, 370115), 'event': 'start'}, {'time': datetime.datetime(2026, 5, 10, 12, 25, 23, 370375), 'event': 'stop'}]
[{'time': datetime.datetime(2026, 5, 10, 12, 25, 23, 370115), 'event': 'start'}, {'time': datetime.datetime(2026, 5, 10, 12, 25, 23, 370375), 'event': 'stop'}, {'time': datetime.datetime(2026, 5, 10, 12, 25, 23, 370555), 'event': 'restart'}]
([{'time': datetime.datetime(2026, 5, 10, 12, 25, 23, 370115), 'event': 'start'}, {'time': datetime.datetime(2026, 5, 10, 12, 25, 23, 370375), 'event': 'stop'}, {'time': datetime.datetime(2026, 5, 10, 12, 25, 23, 370555), 'event': 'restart'}],)


C:\Users\user1\AppData\Local\Temp\ipykernel_22120\1664728694.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  history.append({'time': datetime.utcnow(), 'event': event})


### Solution 11

The default `history` list is shared by every call. Each call appends to the same list stored in `collect_events_bad.__defaults__`.

Correct implementation:

In [16]:
def collect_events_good(event, history=None):
    if history is None:
        history = []
    history.append({'time': datetime.utcnow(), 'event': event})
    return history

print(collect_events_good('start'))
print(collect_events_good('stop'))

custom_history = []
collect_events_good('custom-1', custom_history)
collect_events_good('custom-2', custom_history)
print(custom_history)

[{'time': datetime.datetime(2026, 5, 10, 12, 25, 24, 288392), 'event': 'start'}]
[{'time': datetime.datetime(2026, 5, 10, 12, 25, 24, 288674), 'event': 'stop'}]
[{'time': datetime.datetime(2026, 5, 10, 12, 25, 24, 288862), 'event': 'custom-1'}, {'time': datetime.datetime(2026, 5, 10, 12, 25, 24, 288918), 'event': 'custom-2'}]


C:\Users\user1\AppData\Local\Temp\ipykernel_22120\1132959161.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  history.append({'time': datetime.utcnow(), 'event': event})


## Problem 12 — Production-Style Function Design

Design a function `send_notification` with these requirements:

- `message` is required
- `recipients` is optional and defaults to a fresh empty list
- `created_at` is optional and defaults to the current UTC time
- `metadata` is optional and defaults to a fresh empty dictionary
- explicit `None` for `recipients` should mean no recipients
- explicit `None` for `metadata` should mean no metadata
- omitted values and explicit `None` must be distinguishable

### Solution 12

Because explicit `None` has meaning, use a private sentinel object for omitted values.

In [17]:
_OMITTED = object()

def send_notification(message, *, recipients=_OMITTED, created_at=_OMITTED, metadata=_OMITTED):
    if recipients is _OMITTED:
        recipients = []
    elif recipients is not None:
        recipients = list(recipients)

    if created_at is _OMITTED:
        created_at = datetime.utcnow()

    if metadata is _OMITTED:
        metadata = {}
    elif metadata is not None:
        metadata = dict(metadata)

    return {
        'message': message,
        'recipients': recipients,
        'created_at': created_at,
        'metadata': metadata,
    }

print(send_notification('System online'))
print(send_notification('No recipients', recipients=None))
print(send_notification('No metadata', metadata=None))
print(send_notification('Custom', recipients=['a@example.com'], metadata={'priority': 'high'}))

{'message': 'System online', 'recipients': [], 'created_at': datetime.datetime(2026, 5, 10, 12, 25, 26, 453661), 'metadata': {}}
{'message': 'No recipients', 'recipients': None, 'created_at': datetime.datetime(2026, 5, 10, 12, 25, 26, 453830), 'metadata': {}}
{'message': 'No metadata', 'recipients': [], 'created_at': datetime.datetime(2026, 5, 10, 12, 25, 26, 453922), 'metadata': None}
{'message': 'Custom', 'recipients': ['a@example.com'], 'created_at': datetime.datetime(2026, 5, 10, 12, 25, 26, 454026), 'metadata': {'priority': 'high'}}


C:\Users\user1\AppData\Local\Temp\ipykernel_22120\3922377625.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at = datetime.utcnow()


## Final Best Practices

1. Never use mutable objects like `[]`, `{}`, or `set()` as ordinary default values.
2. Never use dynamic values like `datetime.utcnow()` as defaults when the value should change per call.
3. Use `None` as a simple omission marker only when `None` is not a valid meaningful argument.
4. Use a private sentinel object when `None` is a valid input.
5. Use `is None`, not truthiness checks, when testing for omitted optional values.
6. Copy user-provided mutable inputs if your function must protect itself from outside mutation.
7. Use intentional mutable defaults only when shared state is truly desired and clearly documented.
8. Prefer standard tools like `functools.lru_cache` for caching.